In [1]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Custom State Schema
# We add 'feedback' and 'is_complete' to pass data between the Evaluator and Worker
class State(TypedDict):
    messages: Annotated[list, add_messages]
    feedback: str
    is_complete: bool

In [8]:
# 2. Tools and LLMs Initialization
def simple_search(query: str) -> str:
    """Use this tool to search for factual information."""
    query = query.lower()
    if "use case" in query or "overview" in query or "feature" in query:
        return (
            "LangGraph is ideal for complex, multi-agent workflows and cyclical tasks "
            "like coding assistants. LCEL is designed for simple, linear data pipelines "
            "like basic Retrieval-Augmented Generation (RAG)."
        )
    
    return "LangGraph allows for cyclic execution, unlike standard LangChain Expression Language (LCEL)."

tools = [simple_search]
worker_llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

# Pydantic model forces the Evaluator to output predictable data
class EvaluatorDecision(BaseModel):
    feedback: str = Field(description="Critique of the assistant's answer.")
    is_complete: bool = Field(description="True if the answer fully resolves the user's prompt.")

evaluator_llm = ChatOpenAI(model="gpt-4o-mini").with_structured_output(EvaluatorDecision)

In [9]:
# 3. Define Nodes
def worker_node(state: State):
    messages = state["messages"]
    
    # Inject evaluator feedback if the previous attempt failed
    if state.get("feedback") and not state.get("is_complete"):
        correction_prompt = f"System: Your previous answer was rejected. Feedback: {state['feedback']}. Try again."
        messages = messages + [SystemMessage(content=correction_prompt)]
    
    response = worker_llm.invoke(messages)
    return {"messages": [response]}

def evaluator_node(state: State):
    last_message = state["messages"][-1].content
    user_request = state["messages"][0].content
    
    prompt = f"Original Request: {user_request}\nAssistant Answer: {last_message}\nDoes the answer resolve the request?"
    
    # The LLM returns a structured Pydantic object
    decision = evaluator_llm.invoke(prompt)
    
    # Update the custom state fields
    return {"feedback": decision.feedback, "is_complete": decision.is_complete}

In [10]:
# 4. Define Custom Routing Logic
def worker_router(state: State) -> Literal["tools", "evaluator"]:
    last_message = state["messages"][-1]
    # Route to tools if the LLM made a tool call; otherwise, send to evaluator
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "evaluator"

def evaluator_router(state: State) -> Literal["worker", "END"]:
    # Route to END if criteria met; otherwise, loop back to worker
    if state.get("is_complete"):
        return "END"
    return "worker"

In [11]:
# 5. Build and Compile the Graph
builder = StateGraph(State)

builder.add_node("worker", worker_node)
builder.add_node("tools", ToolNode(tools=tools))
builder.add_node("evaluator", evaluator_node)

builder.add_edge(START, "worker")
builder.add_conditional_edges("worker", worker_router)
builder.add_edge("tools", "worker")
builder.add_conditional_edges("evaluator", evaluator_router, {"worker": "worker", "END": END})

graph = builder.compile()

# Execution Example
inputs = {"messages": [HumanMessage(content="What does LangGraph do differently than LCEL?")]}
for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

{'worker': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 56, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_56f9c4dc61', 'id': 'chatcmpl-DDuX9WVjozqqT54ZqDHB2hMCbVepI', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c9fd3-5ddc-7dc3-96d8-774f15e6cbed-0', tool_calls=[{'name': 'simple_search', 'args': {'query': 'LangGraph features and differences'}, 'id': 'call_PBgZu3d8ha7XZfhOEMsXCQND', 'type': 'tool_call'}, {'name': 'simple_search', 'args': {'query': 'LCEL features and differences'}, 'id': 'call_ySMEFe5F3wbPdlNXsQZzU0L3', 'type': 'tool_call'}], invalid_tool_calls=[], usag

c:\Users\bpu320145\2026_Projects\Agentic_Chatbot\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluatorDecision(feedbac...ail.", is_complete=True), input_type=EvaluatorDecision])
  return self.__pydantic_serializer__.to_python(
